# HallusionBench — Nạp Model từ `.pkl` Bundle & Test Sống Nhiều Case Thực Tế

Notebook riêng biệt, tối giản cho **Google Colab** (hoặc Jupyter local) để:
1. **Nạp lại model** từ file `<model>_bundle.pkl` đã xuất (hoặc tự lấy đúng revision chuẩn từ báo cáo nếu chưa xuất `.pkl`).
2. **Chạy test sống (0 epoch, zero-shot)** trên từng test case có **ảnh thật** của HallusionBench.
3. **Test nhiều nhóm case khác nhau**: cặp đối chứng (ảnh gốc vs ảnh bẫy vs text-only), các ca bất đồng giữa 3 mô hình (`cross_model_disagreements.json`), và ảnh/câu hỏi tự nhập.
4. **Đối chiếu trực tiếp** kết quả vừa chạy sống với `predictions.jsonl` trong báo cáo để khẳng định báo cáo đúng.

## Bước 1 — Kết nối Google Drive & Chuẩn bị Môi trường

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

MODEL_KEY = "smolvlm2"   # Chọn: "smolvlm2" | "qwen3_vl" | "phi4_multimodal"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    REPO_URL = "https://github.com/hoainhan256/vlm-hallucination-benchmark.git"
    BRANCH = "benchmark/hallusionbench"
    REPO_DIR = Path("/content/vlm-hallucination-benchmark")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

    !pip install -q -r requirements.txt
    if MODEL_KEY == "phi4_multimodal":
        !pip uninstall -y -q transformers
        !pip install -q "transformers==4.48.2" "accelerate==1.3.0" "peft==0.13.2" "huggingface-hub>=0.24.0,<1.0" "tokenizers>=0.21,<0.22" backoff soundfile scipy einops
        import transformers
        assert transformers.__version__ == "4.48.2", f"Cần Restart session (Runtime -> Restart session) để nạp transformers 4.48.2 (hiện tại: {transformers.__version__})"
    else:
        !pip install -q --upgrade transformers accelerate qwen-vl-utils num2words

    DRIVE_ROOT = Path("/content/drive/MyDrive/hallusionbench_runs")
    DATA_ROOT = Path("/content/data/HallusionBench")
else:
    REPO_DIR = Path.cwd().resolve()
    if not (REPO_DIR / "benchmarks" / "hallusionbench").exists():
        REPO_DIR = REPO_DIR.parents[1]
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / "tmp" / "hallusionbench_runs"
    DATA_ROOT = Path(os.environ.get("DATA_ROOT", str(REPO_DIR / "tmp" / "HallusionBench")))

BUNDLE_DIR = DRIVE_ROOT / "exported_bundles"
RESULTS_ROOT = REPO_DIR / "benchmarks" / "hallusionbench" / "results"

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Sẵn sàng! MODEL_KEY =", MODEL_KEY)

## Bước 2 — Import `.pkl` Bundle & Nạp Trọng số Model (Zero-Shot, 0 Epoch)

In [ ]:
import json
import pickle
import torch
from benchmarks.hallusionbench.inference import (
    HallusionBenchPromptConfig,
    HallusionRunConfig,
    HuggingFaceHallusionModel,
    MODEL_SPECS,
)

pkl_path = BUNDLE_DIR / f"{MODEL_KEY}_bundle.pkl"
if pkl_path.is_file():
    print("Đang nạp bundle từ file .pkl:", pkl_path)
    with open(pkl_path, "rb") as f:
        bundle = pickle.load(f)
else:
    print("Chưa có file .pkl trên Drive -> Tự lấy đúng revision chuẩn từ run_metadata.json")
    meta = json.loads((RESULTS_ROOT / MODEL_KEY / "run_metadata.json").read_text(encoding="utf-8"))
    bundle = {
        "model_key": MODEL_KEY,
        "model_id": MODEL_SPECS[MODEL_KEY]["model_id"],
        "model_revision": meta["resolved_model_revision"],
        "processor_revision": meta["resolved_processor_revision"],
        "dtype": "float16" if torch.cuda.is_available() else "float32",
        "attn_implementation": "eager" if MODEL_KEY == "phi4_multimodal" else "sdpa",
        "trust_remote_code": MODEL_SPECS[MODEL_KEY]["default_trust_remote_code"],
        "prompt_config": {},
        "generation_config": {"do_sample": False, "max_new_tokens": 128},
        "saved_weights_dir": None,
    }

weights_source = bundle.get("saved_weights_dir") or bundle["model_id"]
use_local_weights = bool(bundle.get("saved_weights_dir") and Path(bundle["saved_weights_dir"]).exists())
runtime_dtype = "float16" if torch.cuda.is_available() else "float32"
is_phi4_gpu = (bundle["model_key"] == "phi4_multimodal" and torch.cuda.is_available())

run_cfg = HallusionRunConfig(
    experiment_id="hb_live_verify",
    run_id=f"live_{MODEL_KEY}",
    model_key=bundle["model_key"],
    model_id=weights_source if use_local_weights else bundle["model_id"],
    stage="pilot",
    model_revision=None if use_local_weights else bundle["model_revision"],
    processor_revision=None if use_local_weights else bundle["processor_revision"],
    dtype=runtime_dtype,
    attn_implementation=bundle["attn_implementation"],
    device_map="auto" if torch.cuda.is_available() else "cpu",
    max_gpu_memory="8GiB" if is_phi4_gpu else None,
    max_cpu_memory="10GiB" if is_phi4_gpu else None,
    offload_folder="/content/phi4_offload" if is_phi4_gpu else None,
    trust_remote_code=bundle["trust_remote_code"],
    prompt_config=HallusionBenchPromptConfig(**bundle.get("prompt_config", {})),
)

model_adapter = HuggingFaceHallusionModel(run_cfg)
model_adapter.load_model()
print(f"Đã nạp xong model {MODEL_KEY} ({'local saved_weights' if use_local_weights else 'HF rev ' + str(bundle['model_revision'])[:10]})!")

## Bước 3 — Nạp Dataset Ảnh Thật & Hàm `test_one_case()`

In [ ]:
import tarfile
import time
import matplotlib.pyplot as plt
from PIL import Image
from huggingface_hub import snapshot_download
from benchmarks.hallusionbench.dataset import HallusionBenchDataset
from benchmarks.hallusionbench.evaluation import parse_yes_no
from benchmarks.hallusionbench.inference import assert_no_label_leakage, prepare_hallusion_input
from vlm_benchmark.results import read_jsonl

HF_DATASET_REPO = "rayguan/HallusionBench"
HF_DATASET_REVISION = "2067dfdce5abc7efdbb0927f4721ae282f0b6c51"
DRIVE_DATASET_TAR = DRIVE_ROOT / "dataset_cache" / f"HallusionBench_{HF_DATASET_REVISION[:8]}.tar"
DRIVE_DATASET_TAR.parent.mkdir(parents=True, exist_ok=True)

if not (DATA_ROOT / "HallusionBench.json").is_file():
    if DRIVE_DATASET_TAR.is_file():
        print("Giải nén dataset có sẵn từ Drive (không cần gọi HF):", DRIVE_DATASET_TAR)
        DATA_ROOT.mkdir(parents=True, exist_ok=True)
        with tarfile.open(DRIVE_DATASET_TAR, "r") as tf:
            tf.extractall(DATA_ROOT)
    else:
        os.environ["HF_HUB_DISABLE_XET"] = "1"
        hf_token = os.environ.get("HF_TOKEN")
        if not hf_token and IN_COLAB:
            try:
                from google.colab import userdata
                hf_token = userdata.get("HF_TOKEN")
            except Exception:
                hf_token = None
        snapshot_download(repo_id=HF_DATASET_REPO, repo_type="dataset", revision=HF_DATASET_REVISION, local_dir=str(DATA_ROOT), token=hf_token)
        with tarfile.open(DRIVE_DATASET_TAR, "w") as tf:
            for item in DATA_ROOT.iterdir():
                if item.name != ".cache":
                    tf.add(item, arcname=item.name)

dataset_index = HallusionBenchDataset(
    json_path=DATA_ROOT / "HallusionBench.json",
    data_root=DATA_ROOT,
).load_index(verify_images=True)

report_preds = {p["sample_id"]: p for p in read_jsonl(RESULTS_ROOT / MODEL_KEY / "predictions.jsonl")}

def test_one_case(sample_id: str, show_image: bool = True) -> dict:
    rec = dataset_index[sample_id]
    prepared = prepare_hallusion_input(rec, data_root=DATA_ROOT, prompt_config=run_cfg.prompt_config, verify_image_exists=True)
    assert_no_label_leakage(prepared, rec)

    t0 = time.perf_counter()
    live_raw = model_adapter.generate(prepared, run_cfg.generation_config_dict())
    elapsed_ms = (time.perf_counter() - t0) * 1000.0

    live_parsed = parse_yes_no(live_raw)
    gt = rec["reference_answer"]
    saved_raw = report_preds[sample_id]["raw_prediction"] if sample_id in report_preds else "(ngoài pilot)"

    if show_image and prepared.visual_input and prepared.image_path:
        plt.figure(figsize=(5, 4))
        plt.imshow(Image.open(prepared.image_path).convert("RGB"))
        plt.title(f"{sample_id} (vis={rec['visual_input_raw']})")
        plt.axis("off")
        plt.show()

    print(f"[{sample_id}] | sub={rec['subcategory']} | vis={rec['visual_input_raw']} | diff={rec.get('difficulty')}")
    print(f"  Prompt      : {prepared.prompt}")
    print(f"  Ground Truth: {gt} ({rec.get('gt_answer_details', '')})")
    print(f"  Live Output : {live_raw!r} -> parsed={live_parsed} ({'ĐÚNG' if live_parsed == gt else 'SAI / ẢO GIÁC'}) [{elapsed_ms:.0f} ms]")
    print(f"  Report Saved: {saved_raw!r}")
    print("-" * 80)
    return {"sample_id": sample_id, "gt": gt, "live_parsed": live_parsed, "live_raw": live_raw, "saved_raw": saved_raw}

## Bước 4 — Test Nhóm Đối Chứng (Không ảnh vs Ảnh gốc vs Ảnh bẫy chỉnh sửa)

In [ ]:
def test_group_live(group_key: str) -> None:
    print(f"\n=== CHẠY TEST SỐNG NHÓM: {group_key} ===")
    sids = [sid for sid in report_preds if dataset_index[sid]["group_or_pair_key"] == group_key]
    for sid in sids:
        test_one_case(sid, show_image=True)

# Chạy thử 2 nhóm điển hình thường gây ảo giác:
test_group_live("VD::illusion::set_12")
test_group_live("VS::chart::set_11")

## Bước 5 — Test Các Ca Bất Đồng trong Báo cáo (`cross_model_disagreements.json`)

In [ ]:
disagreements = json.loads((RESULTS_ROOT / "cross_model_disagreements.json").read_text(encoding="utf-8"))
print(f"Chạy thử 4 ca bất đồng đầu tiên trong tổng số {len(disagreements)} ca:")
for item in disagreements[:4]:
    test_one_case(item["sample_id"], show_image=True)

## Bước 6 — Test Tự do với Ảnh & Câu hỏi Tùy chọn

In [ ]:
from benchmarks.hallusionbench.inference import build_chat_messages, build_prompt, PreparedHallusionInput

def test_custom_image(image_path: str | None, question: str) -> None:
    """Chạy thử model với bất kỳ ảnh & câu hỏi Yes/No nào."""
    has_img = image_path is not None
    prompt = build_prompt(question, prompt_config=run_cfg.prompt_config)
    prepared = PreparedHallusionInput(
        sample_id="custom_case",
        prompt=prompt,
        visual_input=has_img,
        image_path=image_path,
        messages=build_chat_messages(prompt=prompt, visual_input=has_img),
    )
    if has_img:
        plt.figure(figsize=(5, 4))
        plt.imshow(Image.open(image_path).convert("RGB"))
        plt.axis("off")
        plt.show()
    raw = model_adapter.generate(prepared, run_cfg.generation_config_dict())
    print("Prompt :", prompt)
    print("Raw    :", repr(raw))
    print("Parsed :", parse_yes_no(raw))

# Ví dụ: lấy 1 ảnh từ tập HallusionBench và hỏi thử câu khác:
sample_img = dataset_index["catVD_subfigure_set0_fig1_q0_vis2"]["image_path"]
test_custom_image(sample_img, "Are there four people in the image?")